# openAI Harmony

https://cookbook.openai.com/articles/openai-harmony

In [1]:
from openai_harmony import (
    Author,
    Conversation,
    DeveloperContent,
    HarmonyEncodingName,
    Message,
    Role,
    SystemContent,
    StreamableParser,
    ToolDescription,
    load_harmony_encoding,
    ReasoningEffort
)
from mlx_lm import generate, load

In [2]:
checkpoint = "openai/gpt-oss-20b"
model, tokenizer = load(path_or_hf_repo=checkpoint)
encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
stream = StreamableParser(encoding, role=Role.ASSISTANT)

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

In [3]:
system_message = (
    SystemContent.new()
        .with_reasoning_effort(ReasoningEffort.HIGH)
        .with_conversation_start_date("2025-06-28")
)

In [4]:
developer_message = (
    DeveloperContent.new()
        .with_instructions("Always respond in riddles")
        .with_function_tools(
            [
                ToolDescription.new(
                    "get_current_weather",
                    "Gets the current weather in the provided location.",
                    parameters={
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city and state, e.g. San Francisco, CA",
                            },
                            "format": {
                                "type": "string",
                                "enum": ["celsius", "fahrenheit"],
                                "default": "celsius",
                            },
                        },
                        "required": ["location"],
                    },
                ),
            ]
	)
)

In [5]:
messages = [
        Message.from_role_and_content(Role.SYSTEM, system_message),
        Message.from_role_and_content(Role.DEVELOPER, developer_message),
        Message.from_role_and_content(Role.USER, "What is the weather in Tokyo?"),
        Message.from_role_and_content(
            Role.ASSISTANT,
            'User asks: "What is the weather in Tokyo?" We need to use get_current_weather tool.',
        ).with_channel("analysis"),
        Message.from_role_and_content(Role.ASSISTANT, '{"location": "Tokyo"}')
        .with_channel("commentary")
        .with_recipient("functions.get_current_weather")
        .with_content_type("<|constrain|> json"),
        Message.from_author_and_content(
            Author.new(Role.TOOL, "functions.get_current_weather"),
            '{ "temperature": 20, "sunny": true }',
        ).with_channel("commentary"),
    ]

In [6]:
messages

[Message(author=Author(role=<Role.SYSTEM: 'system'>, name=None), content=[SystemContent(model_identity='You are ChatGPT, a large language model trained by OpenAI.', reasoning_effort=<ReasoningEffort.HIGH: 'High'>, conversation_start_date='2025-06-28', knowledge_cutoff='2024-06', channel_config=ChannelConfig(valid_channels=['analysis', 'commentary', 'final'], channel_required=True), tools=None)], channel=None, recipient=None, content_type=None),
 Message(author=Author(role=<Role.DEVELOPER: 'developer'>, name=None), content=[DeveloperContent(instructions='Always respond in riddles', tools={'functions': ToolNamespaceConfig(name='functions', description=None, tools=[ToolDescription(name='get_current_weather', description='Gets the current weather in the provided location.', parameters={'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'The city and state, e.g. San Francisco, CA'}, 'format': {'type': 'string', 'enum': ['celsius', 'fahrenheit'], 'default': 'celsi

In [7]:
convo = Conversation.from_messages(messages)    

In [8]:
convo

Conversation(messages=[Message(author=Author(role=<Role.SYSTEM: 'system'>, name=None), content=[SystemContent(model_identity='You are ChatGPT, a large language model trained by OpenAI.', reasoning_effort=<ReasoningEffort.HIGH: 'High'>, conversation_start_date='2025-06-28', knowledge_cutoff='2024-06', channel_config=ChannelConfig(valid_channels=['analysis', 'commentary', 'final'], channel_required=True), tools=None)], channel=None, recipient=None, content_type=None), Message(author=Author(role=<Role.DEVELOPER: 'developer'>, name=None), content=[DeveloperContent(instructions='Always respond in riddles', tools={'functions': ToolNamespaceConfig(name='functions', description=None, tools=[ToolDescription(name='get_current_weather', description='Gets the current weather in the provided location.', parameters={'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'The city and state, e.g. San Francisco, CA'}, 'format': {'type': 'string', 'enum': ['celsius', 'fahrenheit

In [9]:
tokens = encoding.render_conversation_for_completion(convo, Role.ASSISTANT)

In [10]:
tokens

[200006,
 17360,
 200008,
 3575,
 553,
 17554,
 162016,
 11,
 261,
 4410,
 6439,
 2359,
 22203,
 656,
 7788,
 17527,
 558,
 87447,
 100594,
 25,
 220,
 1323,
 19,
 12,
 3218,
 198,
 6576,
 3521,
 25,
 220,
 1323,
 20,
 12,
 3218,
 12,
 2029,
 279,
 30377,
 289,
 25,
 1932,
 279,
 2,
 13888,
 18403,
 25,
 8450,
 11,
 49159,
 11,
 1721,
 13,
 21030,
 2804,
 413,
 7360,
 395,
 1753,
 3176,
 558,
 63446,
 316,
 1879,
 8437,
 2804,
 810,
 316,
 290,
 49159,
 9334,
 25,
 461,
 44580,
 6120,
 200007,
 200006,
 77944,
 200008,
 2,
 68406,
 279,
 48258,
 9570,
 306,
 151829,
 1032,
 279,
 2,
 20574,
 279,
 877,
 9964,
 279,
 4797,
 9964,
 95359,
 21733,
 290,
 2208,
 11122,
 306,
 290,
 5181,
 5100,
 558,
 2493,
 717,
 23981,
 170154,
 314,
 11350,
 25,
 10168,
 623,
 5030,
 326,
 2608,
 11,
 319,
 1940,
 13,
 6610,
 18826,
 11,
 13180,
 198,
 7693,
 25,
 1621,
 412,
 4078,
 8528,
 392,
 66,
 63110,
 1,
 1022,
 392,
 40364,
 11732,
 672,
 602,
 2787,
 25,
 274,
 63110,
 198,
 9263,
 871,
 1062,

In [11]:
for token in tokens:
    stream.process(token)
    print("--------------------------------")
    print("current_role", stream.current_role)
    print("current_channel", stream.current_channel)
    print("last_content_delta", stream.last_content_delta)
    print("current_content_type", stream.current_content_type)
    print("current_recipient", stream.current_recipient)
    print("current_content", stream.current_content)

--------------------------------
current_role Role.ASSISTANT
current_channel None
last_content_delta None
current_content_type None
current_recipient None
current_content 
--------------------------------
current_role Role.ASSISTANT
current_channel None
last_content_delta None
current_content_type None
current_recipient None
current_content 
--------------------------------
current_role Role.ASSISTANT
current_channel None
last_content_delta None
current_content_type None
current_recipient <|start|>system
current_content 
--------------------------------
current_role Role.ASSISTANT
current_channel None
last_content_delta You
current_content_type None
current_recipient <|start|>system
current_content You
--------------------------------
current_role Role.ASSISTANT
current_channel None
last_content_delta  are
current_content_type None
current_recipient <|start|>system
current_content You are
--------------------------------
current_role Role.ASSISTANT
current_channel None
last_content_del

HarmonyError: unexpected tokens remaining in message header: Some("to=functions.get_current_weather")

In [12]:
response = generate(model=model, tokenizer=tokenizer, prompt=tokens, max_tokens=1024, verbose=True)

<|channel|>final<|message|>In the city of crimson blossoms, the sky whispers a gentle 20 degrees, and the sun smiles upon the streets—no clouds to cloud its bright decree.
Prompt: 240 tokens, 187.179 tokens-per-sec
Generation: 36 tokens, 33.831 tokens-per-sec
Peak memory: 14.611 GB


In [13]:
new_tokens = encoding.encode(response, allowed_special={'<|channel|>', '<|message|>'})

In [14]:
for new_token in new_tokens:
    stream.process(new_token)
    print("--------------------------------")
    print("current_role", stream.current_role)
    print("current_channel", stream.current_channel)
    print("last_content_delta", stream.last_content_delta)
    print("current_content_type", stream.current_content_type)
    print("current_recipient", stream.current_recipient)
    print("current_content", stream.current_content)

HarmonyError: Unexpected token 200005 while expecting start token 200006

In [15]:
parsed_response = encoding.parse_messages_from_completion_tokens(new_tokens, Role.ASSISTANT)

In [16]:
parsed_response

[Message(author=Author(role=<Role.ASSISTANT: 'assistant'>, name=None), content=[TextContent(text='In the city of crimson blossoms, the sky whispers a gentle 20 degrees, and the sun smiles upon the streets—no clouds to cloud its bright decree.')], channel='final', recipient=None, content_type=None)]